In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

print("PyTorch version:", torch.__version__)
print("Libraries imported successfully!")

PyTorch version: 2.11.0
Libraries imported successfully!


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Transforms defined successfully!")

# Resize(224,224) = ResNet18 needs exactly 224x224 images
# ToTensor()      = converts image to numbers (0-1)
# Normalize()     = standardizes pixel values
#                   these specific numbers are 
#                   ImageNet standard values

Transforms defined successfully!


In [6]:
dataset = datasets.ImageFolder(
    root='data/plantvillage/plantvillage dataset/color',
    transform=transform
)

print("Total images:", len(dataset))
print("Number of classes:", len(dataset.classes))
print("Classes:", dataset.classes[:5])

Total images: 54305
Number of classes: 38
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [7]:

from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))
print("Dataloaders created successfully!")

Train size: 43444
Test size: 10861
Dataloaders created successfully!


In [8]:
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace last layer for 38 classes
model.fc = nn.Linear(model.fc.in_features, 38)

# Move model to device
model = model.to(device)

print("Model loaded successfully!")
print("Output classes:", 38)

/opt/anaconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/ahmarpradhan/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


Model loaded successfully!
Output classes: 38


In [9]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Loss and optimizer defined!")

Loss and optimizer defined!


In [10]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {accuracy:.2f}%")

print("Training complete!")

Epoch 1/5 - Loss: 0.5418 - Accuracy: 87.47%
Epoch 2/5 - Loss: 0.1971 - Accuracy: 94.31%
Epoch 3/5 - Loss: 0.1587 - Accuracy: 95.21%
Epoch 4/5 - Loss: 0.1373 - Accuracy: 95.69%
Epoch 5/5 - Loss: 0.1206 - Accuracy: 96.12%
Training complete!


In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 95.64%


In [12]:
torch.save(model.state_dict(), 'models/disease_model.pth')
print("Model saved!")

Model saved!
